# 05 — Fine-tuning wav2vec 2.0 for Egyptian Arabic ASR

CTC fine-tuning of `facebook/wav2vec2-large-xlsr-53` on the scraped Egyptian-Arabic
corpus (see notebooks 01–04 for how the dataset was built). Steps:

1. Normalize transcripts (strip punctuation/tashkeel/latin, unify alef forms) and build a character vocab
2. Build a `Wav2Vec2Processor` (CTC tokenizer + feature extractor) from that vocab
3. On-the-fly audio loading + resampling to 16 kHz in a custom `Dataset`
4. Train with a CTC loss loop; evaluate WER (via `evaluate`) and CER on a held-out split
5. Save model + processor for the FastAPI server in `serve/`

Originally trained on Kaggle (GPU). Expect ~4 epochs to reach **WER ≈ 0.57** on the
held-out split — see `docs/report.pdf` for the full run log.

In [ ]:
# If running on Kaggle/Colab, uncomment:
# !pip install transformers torchaudio librosa datasets evaluate jiwer

## Config

In [ ]:
import os

# === Config ===
DATA_DIR = "../data"                          # dataset root
TRAIN_CSV = f"{DATA_DIR}/train.csv"           # cols: audio, transcript
VAL_CSV = f"{DATA_DIR}/val.csv"
TRAIN_AUDIO_DIR = f"{DATA_DIR}/audio/train"   # wav chunks referenced by the CSVs
VAL_AUDIO_DIR = f"{DATA_DIR}/audio/val"

MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"  # multilingual base — also tried: "facebook/wav2vec2-base"
EPOCHS = 4
BATCH_SIZE = 4        # 8 if VRAM allows
LR = 1e-4
OUTPUT_DIR = "./wav2vec2-egyptian-arabic"
VOCAB_PATH = f"{DATA_DIR}/vocab.json"

## Text normalization & vocabulary

Egyptian-Arabic transcripts need normalization before CTC training: strip
punctuation (incl. Arabic `؟` `،`), Latin characters, tashkeel diacritics, and
unify the alef variants (`إ أ ٱ آ` → `ا`).

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import torch
import torchaudio
import librosa
from torch.utils.data import DataLoader, Dataset
from dataclasses import dataclass
from typing import Union
from transformers import (
    Wav2Vec2ForCTC, Wav2Vec2Processor, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor
)

chars_to_ignore_regex = r'[\\,\?\.\!\-\;\:\'"\u060C\u061F\u201C\u201D\u060D\u0621\u0640]'

def remove_special_characters(text):
    text = re.sub(chars_to_ignore_regex, '', text).lower()
    text = re.sub(r'[a-z]', '', text)                 # drop latin chars
    text = re.sub(r'[إأٱآا]', 'ا', text)               # unify alef forms
    text = re.sub(re.compile(r'[\u064B-\u0652]'), '', text)  # tashkeel
    return text

def build_vocab(csv_files, out_path=VOCAB_PATH):
    vocab = set()
    for csv_path in csv_files:
        data = pd.read_csv(csv_path)
        data['transcript'] = data['transcript'].fillna(" ").astype(str)
        for transcript in data['transcript']:
            vocab.update(remove_special_characters(transcript))

    vocab_dict = {char: idx for idx, char in enumerate(sorted(vocab))}
    vocab_dict['|'] = vocab_dict.get(' ', len(vocab_dict))  # word delimiter
    vocab_dict['[UNK]'] = len(vocab_dict)
    vocab_dict['[PAD]'] = len(vocab_dict)

    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(vocab_dict, f, ensure_ascii=False)
    return vocab_dict

def initialize_processor(vocab_path=VOCAB_PATH):
    tokenizer = Wav2Vec2CTCTokenizer(
        vocab_path, unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|'
    )
    feature_extractor = Wav2Vec2FeatureExtractor(
        feature_size=1, sampling_rate=16000, padding_value=0.0,
        do_normalize=True, return_attention_mask=True
    )
    return Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## Dataset & collator

In [ ]:
class AudioDataset(Dataset):
    """Loads audio + transcript on the fly; resamples everything to 16 kHz."""

    def __init__(self, csv_path, audio_dir, processor, target_sr=16000):
        self.data = pd.read_csv(csv_path).dropna(subset=['audio', 'transcript']).reset_index(drop=True)
        self.data['transcript'] = self.data['transcript'].fillna(" ").astype(str)
        self.audio_dir = audio_dir
        self.processor = processor
        self.target_sr = target_sr

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio_path = os.path.join(self.audio_dir, self.data.iloc[idx]['audio'])
        if not audio_path.endswith('.wav'):
            audio_path += '.wav'
        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")

        speech_array, sampling_rate = torchaudio.load(audio_path)
        speech_array = librosa.resample(
            np.asarray(speech_array[0].numpy()), orig_sr=sampling_rate, target_sr=self.target_sr
        )

        transcript = remove_special_characters(self.data.iloc[idx]['transcript'])
        input_values = self.processor(speech_array, sampling_rate=self.target_sr).input_values[0]
        with self.processor.as_target_processor():
            labels = self.processor(transcript).input_ids

        return {'input_values': input_values, 'labels': labels}


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f['input_values']} for f in features]
        label_features = [{"input_ids": f['labels']} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(label_features, padding=self.padding, return_tensors="pt")
        batch['labels'] = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        return batch


# Build vocab from the training transcripts, then init the processor
vocab_dict = build_vocab([TRAIN_CSV])
processor = initialize_processor()

train_dataset = AudioDataset(TRAIN_CSV, TRAIN_AUDIO_DIR, processor)
val_dataset = AudioDataset(VAL_CSV, VAL_AUDIO_DIR, processor)

data_collator = DataCollatorCTCWithPadding(processor=processor)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=data_collator, pin_memory=True, num_workers=2)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                            collate_fn=data_collator, pin_memory=True, num_workers=2)

## Model

`wav2vec2-large-xlsr-53` — cross-lingual pretrained on 53 languages, so it already
has useful multilingual audio representations; the CTC head is resized to our
Arabic char vocab. The feature extractor stays frozen (standard wav2vec2
fine-tuning practice — the pretrained features are good, we only train the
contextual transformer + CTC head). Dropout + layerdrop for regularization,
`mask_time_prob` for SpecAugment-style masking.

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.1,
    hidden_dropout=0.05,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    gradient_checkpointing=True,          # needed to fit the large model on Kaggle GPUs
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
model.freeze_feature_extractor()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Training loop

CTC loss over variable-length sequences: `input_lengths` = number of output
frames, `target_lengths` = number of non-pad label tokens. Validation computes
loss + WER + CER each epoch.

In [ ]:
from evaluate import load
wer_metric = load('wer')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.CTCLoss()

def calculate_cer(predictions, references):
    """Character error rate via simple positional edit distance."""
    total_cer, total_chars = 0, 0
    for pred, ref in zip(predictions, references):
        distance = sum(1 for a, b in zip(pred, ref) if a != b) + abs(len(pred) - len(ref))
        total_cer += distance
        total_chars += len(ref)
    return total_cer / total_chars if total_chars > 0 else float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_dataloader:
        input_values = batch['input_values'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = model(input_values, labels=labels).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: train loss = {total_loss / len(train_dataloader):.3f}")

    # Validation
    model.eval()
    val_loss, predictions, references = 0, [], []
    with torch.no_grad():
        for batch in val_dataloader:
            input_values = batch['input_values'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_values).logits
            input_lengths = torch.full(
                size=(logits.size(0),), fill_value=logits.size(1), dtype=torch.long
            ).to(device)
            target_lengths = torch.sum(labels != -100, dim=1).to(device)
            val_loss += criterion(
                logits.permute(1, 0, 2), labels, input_lengths, target_lengths
            ).item()

            pred_ids = torch.argmax(logits, dim=-1)
            predictions.extend(processor.batch_decode(pred_ids))
            references.extend(processor.batch_decode(labels, group_tokens=False))

    print(f"  val loss = {val_loss:.3f} | WER = {wer_metric.compute(predictions=predictions, references=references):.3f} | CER = {calculate_cer(predictions, references):.3f}")

# Save for serving
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

## Reported results (original run)

From `docs/report.pdf` — 4 epochs, `wav2vec2-large`, Kaggle GPU:

| Epoch | Train loss | Val loss | WER | CER |
|-------|-----------|----------|-----|-----|
| 1 | 2.49 | 2.24  | 0.78 | 0.97 |
| 2 | 0.99 | 16.97 | 0.64 | 0.96 |
| 3 | 0.80 | 19.03 | 0.60 | 0.96 |
| 4 | 0.71 | 22.96 | 0.57 | 0.96 |

WER improves steadily (0.78 → 0.57) even as val loss diverges — a known CTC
artifact where over-confident logits inflate loss while argmax decoding keeps
getting better. Still, WER 0.57 is far from production: the corpus is small
(~30k noisy auto-labelled utterances), heavily male-skewed (86%), and the Whisper
pseudo-labels carry their own error rate. Future work listed in the report:
more dialects, noise-robustness, better label quality.